# Demo 1 - Detectando URLs maliciosas com Machine Learning

### Cybersecurity MBA - Artificial Intelligence & Robotics Trends (SEGR)

### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

---

**Objetivo desta demo:** construir, do zero, um modelo que olha uma URL e diz se ela parece **phishing/maliciosa** ou **legítima**. Você não precisa ter experiência prévia com IA.

Vamos percorrer 4 etapas, todas com **uma única base de dados**, o Kaggle URL Dataset:

0. Preparar o ambiente
1. Conhecer e padronizar os dados
2. Treinar o primeiro modelo e descobrir que **acurácia alta pode enganar**
3. Melhorar o modelo e olhar por dentro do que ele aprendeu

> Na **Demo 2**, logo em seguida, vamos levar esse mesmo modelo para uma segunda base, a **PhiUSIIL**, e testar se ele aguenta o mundo real.

## Vocabulário rápido (volte aqui sempre que precisar)

| Termo | O que significa | Analogia |
|---|---|---|
| **Dataset** | Tabela com exemplos já conhecidos | Um caderno de casos antigos |
| **Rótulo (label)** | A resposta certa de cada exemplo: malicioso ou legítimo | O gabarito |
| **Treino** | Parte dos dados que o modelo vê para aprender | Os exercícios que o aluno estuda |
| **Teste** | Parte dos dados que o modelo **nunca viu**, usada para avaliar | A prova final |
| **Modelo** | O "cérebro" que resulta do aprendizado | O aluno depois de estudar |
| **Falso negativo** | Ataque real que o modelo chamou de legítimo | O alarme que **não tocou** com o ladrão dentro |
| **Falso positivo** | Site legítimo que o modelo chamou de malicioso | O alarme que tocou à toa |
| **Recall** | Dos ataques reais, quantos o modelo pegou | Quantos ladrões o alarme detectou |
| **Precision** | Dos alertas que o modelo deu, quantos eram ataque de verdade | Quantos toques de alarme eram ladrão de fato |

## Etapa 0 - Preparando o ambiente

In [ ]:
import os, re, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                             confusion_matrix, ConfusionMatrixDisplay)

pd.set_option("display.max_colwidth", 90)
print("Bibliotecas carregadas.")

### Baixando a base de dados

| Base | Ano | Origem | Tamanho |
|---|---|---|---|
| **Kaggle URL Dataset** | ~2015 | Kaggle (`antonyj453/urldataset`), cópia no repositório da disciplina | ~420 mil URLs |

In [ ]:
URL_KAGGLE = "https://raw.githubusercontent.com/ahirtonlopes/AI-and-Robotics-Trends/main/Datasets/url_data.csv"

if not os.path.exists("url_data.csv"):
    urllib.request.urlretrieve(URL_KAGGLE, "url_data.csv")

print("Arquivo pronto:", [f for f in os.listdir(".") if f.endswith(".csv")])

## Etapa 1 - Conhecendo os dados

### 1.1 Conhecendo o Kaggle URL Dataset

Cada linha é uma URL e a sua classificação (`good` ou `bad`).

In [ ]:
kaggle_bruta = pd.read_csv("url_data.csv")
print("Linhas:", len(kaggle_bruta))
kaggle_bruta.sample(8, random_state=7)

### 1.2 Padronizando os dados

Os computadores comparam texto letra por letra. Então `HTTP://WWW.Site.com/` e `site.com` seriam "diferentes". Vamos **padronizar**: tudo em minúsculas, sem `http://`, sem `https://`, sem `www.` e sem a barra final.

Dois cuidados importantes:

- **Rótulo único:** vamos usar `malicioso = 1` para ameaça e `malicioso = 0` para legítimo, nas duas bases.
- **Remover duplicatas:** se a mesma URL aparece no treino e no teste, o modelo "cola" na prova e a nota fica falsamente alta.

> Por que tirar o `https`? Veremos mais adiante: algumas bases têm "atalhos" que fazem o modelo parecer bom sem aprender nada útil sobre segurança.

In [ ]:
def limpar_url(u):
    u = re.sub(r"\s+", "", str(u).lower())      # remove espaços e quebras de linha
    u = re.sub(r"^https?://", "", u)
    u = re.sub(r"^www\.", "", u)
    return u.rstrip("/")

# exemplo
print(limpar_url("HTTPS://www.Banco-Seguro.com/Login/"))

In [ ]:
kaggle = pd.DataFrame({
    "url": kaggle_bruta["url"].map(limpar_url),
    "malicioso": (kaggle_bruta["label"] == "bad").astype(int),
}).drop_duplicates("url")

print(f"Base Kaggle: {len(kaggle):,} URLs (removidas {len(kaggle_bruta) - len(kaggle):,} duplicadas)")

In [ ]:
contagem = kaggle["malicioso"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(4.8, 3.2))
ax.bar(["legítimo", "malicioso"], contagem.values, color=["#4c9a6a", "#c0504d"])
ax.set_title(f"Base Kaggle\n{contagem[1] / len(kaggle):.0%} maliciosas")
ax.set_ylabel("URLs")
plt.tight_layout(); plt.show()

**Observação:** a base é **desbalanceada**: só ~16% das URLs são maliciosas. Isso é realista (na vida real, quase todo o tráfego é legítimo), mas tem uma consequência perigosa que vamos ver já já.

## Etapa 2 - Primeiro modelo (e uma pegadinha)

### 2.1 Transformando texto em números

Modelos de ML só entendem números. A técnica **TF-IDF** transforma cada URL em uma lista de números que indica **quais palavras aparecem nela** e o quão raras/importantes elas são.

Veja como uma URL suspeita é "quebrada em palavras":

In [ ]:
exemplo = "paypa1-secure-login.com/verify/account"

vec_palavras = TfidfVectorizer().fit([exemplo])
print("Palavras que o modelo enxerga:")
print(list(vec_palavras.get_feature_names_out()))

### 2.2 Separando treino e teste (a regra 80/20)

Guardamos **20% dos dados escondidos** como "prova final". O modelo estuda com os outros 80% e só depois é avaliado nos 20% que nunca viu. O `stratify` garante que a proporção de maliciosas seja a mesma nos dois grupos.

Vamos criar uma função que faz o processo completo, para reaproveitar várias vezes:

In [ ]:
def treinar(df, analisador="word", balanceado=False, seed=42):
    """Treina um modelo e devolve tudo que precisamos para avaliar."""
    treino, teste = train_test_split(df, test_size=0.2, random_state=seed, stratify=df["malicioso"])

    if analisador == "word":      # olha palavras inteiras
        vec = TfidfVectorizer(sublinear_tf=True)
    else:                         # olha pedaços de 2 a 4 letras
        vec = TfidfVectorizer(analyzer="char", ngram_range=(2, 4), max_features=300000, sublinear_tf=True)

    X_treino = vec.fit_transform(treino["url"])
    X_teste = vec.transform(teste["url"])

    modelo = LogisticRegression(max_iter=1000, class_weight="balanced" if balanceado else None)
    modelo.fit(X_treino, treino["malicioso"])

    return {"modelo": modelo, "vec": vec, "teste": teste, "X_teste": X_teste,
            "pred": modelo.predict(X_teste)}


def metricas(y_real, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred).ravel()
    return {"acurácia": accuracy_score(y_real, y_pred),
            "recall": recall_score(y_real, y_pred),
            "precision": precision_score(y_real, y_pred),
            "ataques que passaram (FN)": fn,
            "alarmes falsos (FP)": fp}

### 2.3 Treinando o modelo (a versão "clássica" da aula)

Usamos a **Regressão Logística**, um algoritmo simples e muito usado como ponto de partida.

In [ ]:
r_base = treinar(kaggle, analisador="word", balanceado=False)
y_real = r_base["teste"]["malicioso"]

print(f"Acurácia do modelo: {accuracy_score(y_real, r_base['pred']):.1%}")

**~96% de acurácia!** Parece excelente, certo? Antes de comemorar, vamos olhar **onde** o modelo errou.

### 2.4 A matriz de confusão

Ela mostra todos os acertos e erros do modelo em uma tabelinha 2x2.

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 4))
ConfusionMatrixDisplay.from_predictions(
    y_real, r_base["pred"], display_labels=["legítimo", "malicioso"],
    cmap="Blues", values_format=",", ax=ax, colorbar=False)
ax.set_title("O que o modelo acertou e errou")
ax.set_xlabel("O modelo disse"); ax.set_ylabel("A realidade era")
plt.tight_layout(); plt.show()

m = metricas(y_real, r_base["pred"])
total_ataques = int((y_real == 1).sum())
print(f"Dos {total_ataques:,} ataques reais no teste, o modelo deixou passar {m['ataques que passaram (FN)']:,} "
      f"({1 - m['recall']:.0%}).")
print(f"Recall: {m['recall']:.0%}  |  Precision: {m['precision']:.0%}")

### A pegadinha

A **acurácia** (~96%) conta todos os acertos juntos. Como ~84% das URLs são legítimas, um modelo que acertasse todas as legítimas e errasse boa parte dos ataques ainda teria nota altíssima.

Em segurança, o que importa é o **recall**: **quantos ataques reais eu peguei?** Cada ataque que passa é um incidente em potencial.

> **Lição 1:** em cibersegurança, nunca avalie um modelo só pela acurácia.

## Etapa 3 - Melhorando o modelo

Vamos fazer duas melhorias e medir o efeito de cada uma.

**Melhoria A - Dar mais peso aos ataques (`class_weight="balanced"`)**
Como os ataques são raros, o modelo tende a ignorá-los. Com esse ajuste, errar um ataque "dói mais" no treino do que errar um site legítimo.

**Melhoria B - Olhar pedaços de letras, não só palavras**
Atacantes escrevem `paypa1`, `amaz0n`, `secure-login-verify`. Palavras inteiras não capturam isso. Se olharmos **pedaços de 2 a 4 caracteres** (`pa`, `ay`, `yp`, `pa1`, `1-s`...), o modelo percebe padrões típicos de URLs falsas.

In [ ]:
vec_car = TfidfVectorizer(analyzer="char", ngram_range=(2, 4)).fit([exemplo])
pedacos = list(vec_car.get_feature_names_out())
print(f"A mesma URL vira {len(pedacos)} pedaços. Alguns deles:")
print(pedacos[:30])

In [ ]:
r_A = treinar(kaggle, analisador="word", balanceado=True)
r_B = treinar(kaggle, analisador="char", balanceado=True)

tabela = pd.DataFrame({
    "1. Base (palavras)":                 metricas(y_real, r_base["pred"]),
    "2. + pesos balanceados (A)":         metricas(y_real, r_A["pred"]),
    "3. + pedaços de letras (A + B)":     metricas(y_real, r_B["pred"]),
}).T

tabela.style.format({"acurácia": "{:.1%}", "recall": "{:.1%}", "precision": "{:.1%}",
                     "ataques que passaram (FN)": "{:,.0f}", "alarmes falsos (FP)": "{:,.0f}"})

Repare no que aconteceu: o **recall subiu** (pegamos mais ataques) e o número de ataques que passaram caiu bastante. Em troca, aparecem mais **alarmes falsos**. Esse é o **dilema clássico da segurança**: pegar mais ameaças custa mais falsos alertas para o time investigar.

> **Lição 2:** melhorar um modelo quase sempre é um *trade-off*. Nós é que decidimos qual erro é mais caro.

### Olhando por dentro: o que o modelo aprendeu?

A Regressão Logística é "transparente": dá para ver quais pedaços de texto mais empurram uma URL para o lado **malicioso** ou **legítimo**.

In [ ]:
nomes = np.array(r_B["vec"].get_feature_names_out())
pesos = r_B["modelo"].coef_[0]
ordem = np.argsort(pesos)
legiveis = [i for i in ordem if len(nomes[i]) >= 4]     # só pedaços com 4 letras ou mais, mais fáceis de ler

print("Pedaços que mais indicam MALICIOSO:", list(nomes[legiveis[-15:]][::-1]))
print()
print("Pedaços que mais indicam LEGÍTIMO: ", list(nomes[legiveis[:15]]))

## O que aprendemos nesta demo

1. **Acurácia engana.** Em segurança, olhe recall, precision e a matriz de confusão.
2. **Melhorar é escolher.** Mais recall custa mais alarmes falsos. Quem decide qual erro é mais caro somos nós.
3. **Dá para olhar por dentro.** Um modelo simples como a Regressão Logística mostra quais pedaços de texto ele usa para decidir.

### E agora?

Nosso melhor modelo tem **recall acima de 95%** no Kaggle URL Dataset. Parece pronto para produção, certo?

Na **Demo 2**, vamos fazer a pergunta que importa: **ele funciona com URLs de outra origem, a base PhiUSIIL?**